# M61 / TNG50-1 SID 488530: alpha=5 noflip oriented H I and velocity analysis

This notebook runs the yt-based oriented H I column, LOS velocity, and annular rotation-curve workflow using the saved L4Rvir ray recipe row for `J122138+043026`, `alpha=5`, `mode=noflip`.

In [ ]:
# Cell 1: imports, paths, config
from pathlib import Path
import json
import sys

from IPython.display import Image, display

PROJECT_ROOT = Path('/home/tsingh65/m61-tng')
SCRIPT_DIR = PROJECT_ROOT / 'scripts'
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

import m61_oriented_HI_vlos_rotation_alpha5 as m61

config = m61.AnalysisConfig(
    npix=2000,
    bootstrap_samples=200,
    logN_HI_min=17.0,
)
paths = m61.setup_output_dirs(config)
m61.setup_logging(paths)

print('Output directory:', paths['out'])
print('Cutout path:', config.preferred_cutout_path)
print('Recipe CSV:', config.recipe_csv)

In [ ]:
# Cell 2: helper functions
def read_json(path):
    with Path(path).open() as f:
        return json.load(f)

def show_figure(name, width=850):
    path = paths['figures'] / name
    print(path)
    display(Image(filename=str(path), width=width))

print(config)

In [ ]:
# Cell 3: load recipe and dataset, define fields
geometry = m61.load_recipe_and_geometry(config, paths)
ds, field_info = m61.load_dataset_and_define_fields(config, geometry, paths)

print('Selected cutout:', field_info['cutout_h5'])
print('LOS:', geometry['los_hat'])
print('dot(saved_los, p1-p0):', geometry['dot_saved_los_with_p1_minus_p0'])
print('Anchor ckpc/h:', geometry['anchor_ckpch'])
print('Systemic velocity:', field_info['systemic_velocity'])

In [ ]:
# Cell 4: make projections and save maps
maps = m61.make_projections(config, ds, geometry, paths)

print('Projection stats:')
print(json.dumps(m61.json_sanitize(maps['stats']), indent=2))

In [ ]:
# Cell 5: run rotation curve / annuli analysis
rotation_table, ism_velocity_at_rho = m61.run_rotation_curve_analysis(config, maps, geometry, paths)

metadata = {
    'config': m61.asdict(config),
    'paths': {key: str(value) for key, value in paths.items()},
    'geometry': geometry,
    'field_info': field_info,
    'projection_stats': maps['stats'],
    'projection_settings': {
        'center_ckpch_code_length': geometry['anchor_ckpch'],
        'normal_vector': geometry['normal_vector'],
        'north_vector': geometry['e2_hat'],
        'width_kpc': config.width_kpc,
        'width_code_length_ckpch': config.width_kpc * m61.H_TNG,
        'npix': config.npix,
    },
    'ism_velocity_at_rho': ism_velocity_at_rho,
}
m61.write_json(paths['data'] / 'analysis_metadata_alpha5_noflip.json', metadata)

rotation_table.head(), rotation_table.tail()

In [1]:
# Cell 6: print final v_ISM at rho and show saved figures
print(json.dumps(m61.json_sanitize(ism_velocity_at_rho), indent=2))

show_figure('HI_column_density_projection_alpha5_noflip.png')
show_figure('vlos_HI_weighted_projection_alpha5_noflip.png')
show_figure('vlos_gas_density_weighted_projection_alpha5_noflip.png')
show_figure('rotation_curve_alpha5_noflip_HIweighted.png')
show_figure('annulus_geometry_overlay_alpha5_noflip.png')

NameError: name 'json' is not defined